# Atlas-400F (notional cargo jet) — requirements-first program + report

Walkthrough: **L1 specs** (stdlib) **→** nested `RequirementBlock` + `allocate(..., inputs=…)` **→** composed **`Aircraft`** **→** roll-ups and **two fake external tools** **→** `extract` **→** ASCII snapshot.

**Import path:** package at `thundergraph-model/examples/commercial_aircraft/`. The next code cell prepends `thundergraph-model/` (for `tg_model`) and `examples/` (for `commercial_aircraft`). Unpackaged demos mean this `sys.path` step — restart the kernel after API changes.


## Showcase thesis (two independent tracks — be honest)

1. **Mission desk (toy):** `AtlasMissionDesk` scales a **baseline max range** with payload and subtracts the **requested design range** → `mission_range_margin_m`. `mission_range_margin_non_negative` is **only** about this toy, not certification.
2. **Declared envelope:** Program constraints compare **scenario** payload/range to **rolled-up masses** and **parameters** you set (`modeled_max_design_range_m`, MTOW/MZFW-style book). This track does **not** use the desk’s physics.

A **PASS** means both tracks agree for this parameter set — **not** a single integrated performance model. See the report banner *“Thesis (read this once)”* for the same wording.

**L1 table:** `verification` column comes from `l1_specs` — `executable_acceptance` vs `evidenced_by_constraints` vs `context_citations_only` (regulatory/methodology framing, not a cert claim).

**Scope:** v1 is a ThunderGraph API slice (composition, `parameter_ref`, external compute at two levels). It is **not** the full discipline matrix in IMPLEMENTATION_PLAN.md.


## Composition (high level)

| Layer | Role |
|-------|------|
| `CargoJetProgram` | Root `System`: scenario parameters, citations, L1 blocks, mission desk → `mission_range_margin_m`, envelope constraints |
| `Aircraft` | Vehicle part: assembly tree, OEW roll-up, `modeled_max_payload_kg` |
| Major assemblies | Stub dry masses + `WingAssembly` structural intensity (second external tool) |


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def _thundergraph_pkg_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(24):
        if (p / "tg_model" / "__init__.py").is_file():
            return p
        nested = p / "thundergraph-model"
        if (nested / "tg_model" / "__init__.py").is_file():
            return nested
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()


_cwd = Path.cwd().resolve()
_pkg_root = _thundergraph_pkg_root(_cwd)
_examples = _pkg_root / "examples"
for _p in (_pkg_root, _examples):
    _s = str(_p)
    if _s in sys.path:
        sys.path.remove(_s)
    sys.path.insert(0, _s)

print("tg_model root:", _pkg_root)
print("examples on path:", _examples)


tg_model root: /Users/lmikhe/UserApps/ThunderGraph/backend-monorepo/thundergraph-model
examples on path: /Users/lmikhe/UserApps/ThunderGraph/backend-monorepo/thundergraph-model/examples


## Nominal scenario (inside envelope, desk has margin)

Notional weight book: **140 t OEW**, **240 t MZFW** → **100 t** structural payload cap; **95 t** scenario payload. **8,000 km** design range vs **9,000 km** modeled max and a **10,000 km** desk baseline so the toy margin stays positive.

Knobs that move the verdict first: **`mission_desk_baseline_max_range_m`** (desk), **`scenario_design_range_m`**, **`modeled_max_design_range_m`** (envelope), and assembly dry masses (roll-up / MTOW closure).


In [2]:
from unitflow import Quantity
from unitflow.catalogs.si import kg, m

from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types
from commercial_aircraft.reporting import extract_cargo_jet_evaluation_report, format_cargo_jet_report

from tg_model.execution.configured_model import instantiate
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph


def run_case(label: str, inputs: dict) -> None:
    reset_commercial_aircraft_types()
    cm = instantiate(CargoJetProgram)
    ac = cm.aircraft
    graph, handlers = compile_graph(cm)
    v = validate_graph(graph)
    assert v.passed, v.failures
    ctx = RunContext()
    result = Evaluator(graph, compute_handlers=handlers).evaluate(ctx, inputs=inputs)
    report = extract_cargo_jet_evaluation_report(cm, ctx, result)
    print(f"\n{'=' * 72}\nCASE: {label}\n{'=' * 72}")
    print(format_cargo_jet_report(report))
    print(f"\nEvaluator passed: {result.passed}")
    if result.failures:
        for line in result.failures:
            print("  failure:", line)


def nominal_inputs(cm, ac):
    return {
        cm.scenario_payload_mass_kg.stable_id: Quantity(95_000, kg),
        cm.scenario_design_range_m.stable_id: Quantity(8_000_000, m),
        cm.mission_desk_baseline_max_range_m.stable_id: Quantity(10_000_000, m),
        ac.modeled_max_design_range_m.stable_id: Quantity(9_000_000, m),
        ac.notional_mzfw_kg.stable_id: Quantity(240_000, kg),
        ac.notional_mtow_kg.stable_id: Quantity(280_000, kg),
        ac.notional_trip_fuel_kg.stable_id: Quantity(40_000, kg),
        ac.fuselage.dry_mass_kg.stable_id: Quantity(45_000, kg),
        ac.wing.dry_mass_kg.stable_id: Quantity(32_000, kg),
        ac.wing.notional_wing_span_m.stable_id: Quantity(64, m),
        ac.empennage.dry_mass_kg.stable_id: Quantity(8_000, kg),
        ac.landing_gear.dry_mass_kg.stable_id: Quantity(12_000, kg),
        ac.propulsion_installation.dry_mass_kg.stable_id: Quantity(28_000, kg),
        ac.aircraft_systems.dry_mass_kg.stable_id: Quantity(15_000, kg),
    }


reset_commercial_aircraft_types()
_cm = instantiate(CargoJetProgram)
_ac = _cm.aircraft
run_case("nominal (expect PASS)", nominal_inputs(_cm, _ac))



CASE: nominal (expect PASS)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): PASS
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): 2077236.6898835376 m
  Margin (rounded): ~2,077.2 km
  Margin ≥ 0: True
Declared envelope track (MTOW / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: True

Scenario inputs
quantity                          | value      | human (length)            
----------------------------------+------------+

## Stress case (desk fails; envelope may still pass)

Drop **`mission_desk_baseline_max_range_m`** to **6,000 km** while keeping an **8,000 km** requested range. The **toy margin** goes negative → `mission_range_margin_non_negative` **FAIL**. Envelope constraints can still **PASS**, which shows the two tracks are **not** the same check.


In [3]:
reset_commercial_aircraft_types()
cm2 = instantiate(CargoJetProgram)
ac2 = cm2.aircraft
stress = nominal_inputs(cm2, ac2)
stress[cm2.mission_desk_baseline_max_range_m.stable_id] = Quantity(6_000_000, m)
run_case("stressed desk baseline (expect FAIL on desk track)", stress)



CASE: stressed desk baseline (expect FAIL on desk track)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): FAIL
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): -1953657.9860698776 m
  Margin (rounded): ~-1,953.7 km
  Margin ≥ 0: False
Declared envelope track (MTOW / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: False

Scenario inputs
quantity                          | value     | human (length)          
------------------